[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/04-machine-learning/ml-bootstrap.ipynb)

# Bootstrap Resampling & Confidence Intervals

*AIBits Academy · Machine Learning End To End · Statistical Foundations · New*

You've already used the bootstrap once, implicitly, inside Random Forest's bagging. This chapter generalises it into a tool for quantifying uncertainty around any statistic — a metric, a coefficient, even a whole model's prediction.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

## The Core Idea

You typically have one dataset and one estimate (a mean, an R², a model's accuracy on a test set). But how much would that estimate vary if you'd happened to collect a slightly different sample? Without collecting new data, the bootstrap answers this by **resampling with replacement** from the data you already have, treating your sample as a stand-in for the true population:

- Draw a new sample of size n *with replacement* from your original n observations (some points appear multiple times, others not at all — on average ~63.2% of unique points are included, exactly the OOB fraction from Random Forest)

- Compute your statistic of interest (mean, R², accuracy, a specific coefficient) on this resample

- Repeat B times (typically 1,000–10,000), building up a distribution of the statistic

- Use the spread of that distribution directly as your uncertainty estimate

## Bootstrap Confidence Interval — the Percentile Method

$$95\%\ \mathrm{CI} = \left[\, 2.5\text{th percentile of bootstrap distribution},\ \ 97.5\text{th percentile of bootstrap distribution} \,\right]$$

No formula for the statistic's exact sampling distribution is required — this works identically whether the statistic is a simple mean (which has a well-known formula) or something with no clean closed-form variance at all, like a Random Forest's feature importance or the median.

## Code — Bootstrapping a Confidence Interval for Mean Order Value

In [ ]:
import numpy as np

# Swiggy order values (₹) — a skewed, non-normal distribution (a handful of large bulk orders)
np.random.seed(9)
orders = np.concatenate([np.random.normal(350,80,480), np.random.normal(2200,400,20)])

def bootstrap_ci(data, statistic=np.mean, B=5000, ci=0.95):
    n = len(data)
    boot_stats = np.empty(B)
    for i in range(B):
        resample = np.random.choice(data, size=n, replace=True)
        boot_stats[i] = statistic(resample)
    lower = np.percentile(boot_stats, (1-ci)/2*100)
    upper = np.percentile(boot_stats, (1-(1-ci)/2)*100)
    return statistic(data), lower, upper

point_est, lo, hi = bootstrap_ci(orders, statistic=np.mean)
print(f"Mean order value: ₹{point_est:.1f}   95% CI: [₹{lo:.1f}, ₹{hi:.1f}]")

# The bootstrap works identically for statistics with NO clean formula, like the median
point_med, lo_med, hi_med = bootstrap_ci(orders, statistic=np.median)
print(f"Median order value: ₹{point_med:.1f}   95% CI: [₹{lo_med:.1f}, ₹{hi_med:.1f}]")

Notice the mean's confidence interval is much wider than the median's — the mean is far more sensitive to the handful of large bulk orders in the tail, and the bootstrap correctly reflects that extra instability without you having to derive it analytically.

## Try It — Watch the Bootstrap Distribution Build, Draw by Draw

This is the exact 500-order Swiggy dataset from the code above (same seed, same generation). Each click below draws one real bootstrap resample — 500 draws with replacement from this data — computes its mean, and adds it to the running histogram. Watch the live 95% interval (2.5th/97.5th percentile of everything drawn so far) converge as you add more resamples.

## Bootstrapping a Model Metric — Is 91% Accuracy Actually Better Than 89%?

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X, y = make_classification(n_samples=600, n_features=8, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)
clf = RandomForestClassifier(random_state=42).fit(X_tr, y_tr)
preds = clf.predict(X_te)

# Bootstrap the TEST SET (not the training data) to get a CI on the accuracy estimate itself
n_test = len(y_te)
boot_accs = []
for _ in range(3000):
    idx = np.random.choice(n_test, size=n_test, replace=True)
    boot_accs.append((preds[idx] == y_te[idx]).mean())
lo, hi = np.percentile(boot_accs, [2.5, 97.5])
print(f"Test accuracy: {(preds==y_te).mean():.3f}   95% CI: [{lo:.3f}, {hi:.3f}]")
# If a competing model's accuracy falls WITHIN this interval, the difference may just be noise

A rival model scoring 89% sits comfortably inside this interval — the apparent 2-point gap could easily be sampling noise from this particular 180-row test set, not a genuine difference in model quality. Without the bootstrap CI, it's tempting to declare 91% the winner; with it, the honest conclusion is "not distinguishable at this sample size."

## Where You've Already Seen This Idea

| Earlier concept | Bootstrap connection |
|---|---|
| Random Forest's bagging | Each tree trains on one bootstrap resample of the training data |
| Out-of-Bag (OOB) error | The ~36.8% of points excluded from a given resample — the same complement fraction shown above |
| k-Fold Cross-Validation | A different resampling strategy (partition, not resample-with-replacement) for estimating generalisation error |

> **🔗 Real-World Link — Screen Time Analysis**
>
> 54 real days of personal Instagram/WhatsApp screen-time logs (31 vs. 99 minutes/day on average) — a natural candidate for bootstrapping a confidence interval around true average daily usage per app. [See the case study →](https://statso.io/screen-time-analysis-case-study/) ·

### ❓ Conceptual Q&A

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · A bootstrap confidence interval

Resample `orders` with replacement 3,000 times (`rng.choice`), take each resample's mean, and store the 2.5th and 97.5th percentiles in `lo` and `hi`.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
orders = np.concatenate([rng.normal(350, 80, 200), rng.normal(2200, 400, 8)])
lo = hi = None   # TODO


In [ ]:
try:
    check("interval contains the sample mean", lo < orders.mean() < hi)
    check("interval is not absurdly wide", hi - lo < 150)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
rng = np.random.default_rng(0)
orders = np.concatenate([rng.normal(350, 80, 200), rng.normal(2200, 400, 8)])
means = [rng.choice(orders, size=len(orders), replace=True).mean() for _ in range(3000)]
lo, hi = np.percentile(means, [2.5, 97.5])

```

</details>

### Exercise 2 · Medium · Bootstrap standard error

The bootstrap works for **any** statistic. Write `boot_se(data, stat, B=2000, seed=0)` returning the standard deviation of `stat` over `B` resamples. Compute it for the mean (`se_mean`) and the median (`se_median`). For the mean it should be close to `std/sqrt(n)`.

In [ ]:
import numpy as np
def boot_se(data, stat, B=2000, seed=0):
    pass   # TODO
se_mean = se_median = None


In [ ]:
try:
    check("mean SE matches the formula", abs(se_mean - orders.std(ddof=1) / np.sqrt(len(orders))) / se_mean < 0.15)
    check("median SE computed", se_median > 0)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
def boot_se(data, stat, B=2000, seed=0):
    r = np.random.default_rng(seed)
    return float(np.std([stat(r.choice(data, len(data))) for _ in range(B)]))
se_mean, se_median = boot_se(orders, np.mean), boot_se(orders, np.median)

```

</details>

### Exercise 3 · Stretch · Bootstrap a correlation

Bootstrap the Pearson correlation between `ad` and `sales`: resample **row pairs** together (indices), 2,000 times. Store the 95% percentile interval in `ci`.

In [ ]:
import numpy as np
rng = np.random.default_rng(1)
ad = rng.uniform(1, 10, 60)
sales = 5 + 2 * ad + rng.normal(0, 4, 60)
ci = None   # TODO


In [ ]:
try:
    r = np.corrcoef(ad, sales)[0, 1]
    check("interval contains the sample correlation", ci[0] < r < ci[1])
    check("valid correlation bounds", -1 <= ci[0] < ci[1] <= 1)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
rng = np.random.default_rng(1)
ad = rng.uniform(1, 10, 60)
sales = 5 + 2 * ad + rng.normal(0, 4, 60)
rs = []
for _ in range(2000):
    i = rng.integers(0, 60, 60)
    rs.append(np.corrcoef(ad[i], sales[i])[0, 1])
ci = tuple(np.percentile(rs, [2.5, 97.5]))

```

Resampling the pairs (not the columns separately) preserves the relationship being measured.

</details>

---
*Back to the course: **Machine Learning End To End → Bootstrap Resampling & Confidence Intervals**.*